In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Path to the cleaned data folder
data_path = '../data/cleaned/'

policies_df = pd.read_csv(f'{data_path}policies_cleaned.csv')
claims_df = pd.read_csv(f'{data_path}claims_cleaned.csv')
customers_df = pd.read_csv(f'{data_path}customers_cleaned.csv')
vehicles_df = pd.read_csv(f'{data_path}vehicles_cleaned.csv')
payments_df = pd.read_csv(f'{data_path}payments_cleaned.csv')

print("Policies shape:", policies_df.shape)
policies_df.head()

Policies shape: (25000, 12)


,policy_id,customer_id,vehicle_id,policy_start_date,policy_end_date,policy_type,premium_amount,coverage_amount,policy_status,payment_frequency,policy_duration_days,policy_active_flag
0,POL_000001,CUST_000216,VEH_000001,2026-04-10,2027-04-09,Comprehensive,18799.02,572309.02,Active,Annual,364,1
1,POL_000002,CUST_011180,VEH_000002,2024-09-30,2025-09-29,Comprehensive,24413.33,812763.96,Renewed,Semi-Annual,364,0
2,POL_000003,CUST_012391,VEH_000003,2024-01-30,2025-01-29,Comprehensive,27496.06,738543.68,Expired,Annual,365,0
3,POL_000004,CUST_005106,VEH_000004,2023-10-25,2024-10-24,Comprehensive,16152.26,588084.92,Renewed,Annual,365,0
4,POL_000005,CUST_008384,VEH_000005,2024-03-27,2025-03-26,Third Party,7510.84,961163.46,Renewed,Annual,364,0


How many policies are active, expired, cancelled, and renewed?

In [5]:
policy_status_counts = policies_df['policy_status'].value_counts()
policy_status_pct = policies_df['policy_status'].value_counts(normalize=True) * 100

policy_status_summary = pd.DataFrame({
    'Policy Count': policy_status_counts,
    'Percentage (%)': policy_status_pct.round(2)
})

display(policy_status_summary)

,Policy Count,Percentage (%)
policy_status,,
Expired,10464,41.86
Active,7551,30.20
Renewed,6665,26.66
Cancelled,320,1.28


Which policy types generate the most premium?

In [6]:
# Gross Written Premium by Policy Type
premium_summary = policies_df.groupby('policy_type')['premium_amount'].agg(
    Total_Premium='sum',
    Average_Premium='mean',
    Policy_Count='count'
).sort_values(by='Total_Premium', ascending=False)

premium_summary['Total_Premium (M INR)'] = (premium_summary['Total_Premium'] / 1e6).round(2)
premium_summary['Average_Premium (INR)'] = premium_summary['Average_Premium'].round(2)

display(premium_summary[['Policy_Count', 'Total_Premium (M INR)', 'Average_Premium (INR)']])

,Policy_Count,Total_Premium (M INR),Average_Premium (INR)
policy_type,,,
Comprehensive,15085,544.96,36126.19
Zero Depreciation,3700,175.08,47318.45
Third Party,6215,75.53,12152.36


How many claims are registered, approved, rejected, and pending?

In [7]:
# Claim status volume breakdown
claim_status_counts = claims_df['claim_status'].value_counts()
claim_status_pct = claims_df['claim_status'].value_counts(normalize=True) * 100

claim_status_summary = pd.DataFrame({
    'Claim Count': claim_status_counts,
    'Percentage (%)': claim_status_pct.round(2)
})

display(claim_status_summary)

,Claim Count,Percentage (%)
claim_status,,
Approved,3790,37.90
Settled,3259,32.59
Pending,1791,17.91
Rejected,1160,11.60


What is the total and average claim amount?

In [8]:
# Aggregate claim cost metrics
total_claim = claims_df['claim_amount'].sum()
avg_claim = claims_df['claim_amount'].mean()
median_claim = claims_df['claim_amount'].median()

claim_metrics = pd.DataFrame({
    'Metric': ['Total Claim Amount', 'Average Claim Amount', 'Median Claim Amount'],
    'Value (INR)': [f"₹{total_claim:,.2f}", f"₹{avg_claim:,.2f}", f"₹{median_claim:,.2f}"]
})

display(claim_metrics)

,Metric,Value (INR)
0,Total Claim Amount,"₹2,885,436,440.65"
1,Average Claim Amount,"₹288,543.64"
2,Median Claim Amount,"₹205,040.10"


Which vehicle types generate more claims?

In [9]:
# Merge claims with policy and vehicle details
claims_veh = claims_df.merge(policies_df[['policy_id', 'vehicle_id']], on='policy_id', how='left')
claims_veh = claims_veh.merge(vehicles_df[['vehicle_id', 'vehicle_type']], on='vehicle_id', how='left')

vehicle_claims = claims_veh.groupby('vehicle_type').agg(
    Claim_Count=('claim_id', 'count'),
    Total_Claim_Amount=('claim_amount', 'sum'),
    Average_Claim_Amount=('claim_amount', 'mean')
).sort_values(by='Claim_Count', ascending=False)

vehicle_claims['Total_Claim_Amount (M INR)'] = (vehicle_claims['Total_Claim_Amount'] / 1e6).round(2)
vehicle_claims['Average_Claim_Amount (INR)'] = vehicle_claims['Average_Claim_Amount'].round(2)

display(vehicle_claims[['Claim_Count', 'Total_Claim_Amount (M INR)', 'Average_Claim_Amount (INR)']])

,Claim_Count,Total_Claim_Amount (M INR),Average_Claim_Amount (INR)
vehicle_type,,,
Car,10000,2885.44,288543.64


Which regions have higher claim frequency or claim severity?

In [10]:
# Join claims with customer state information
claims_geo = claims_df.merge(customers_df[['customer_id', 'state']], on='customer_id', how='left')

region_metrics = claims_geo.groupby('state').agg(
    Claim_Frequency=('claim_id', 'count'),
    Total_Claims=('claim_amount', 'sum'),
    Average_Severity=('claim_amount', 'mean')
)

print("Top 5 States by Claim Frequency:")
display(region_metrics.sort_values(by='Claim_Frequency', ascending=False)[['Claim_Frequency']].head(5))

print("\nTop 5 States by Claim Severity (Average Claim Amount):")
display(region_metrics.sort_values(by='Average_Severity', ascending=False)[['Average_Severity']].round(2).head(5))

Top 5 States by Claim Frequency:


,Claim_Frequency
state,
Maharashtra,1178
Telangana,1164
Tamil Nadu,884
Gujarat,883
Karnataka,870



Top 5 States by Claim Severity (Average Claim Amount):


,Average_Severity
state,
West Bengal,317080.82
Tamil Nadu,303987.54
Andhra Pradesh,301907.83
Kerala,301382.84
Gujarat,301233.86


Which customer segments generate higher claim values?

In [11]:
# Segment customers by age groups and evaluate total claim values
claims_cust = claims_df.merge(customers_df[['customer_id', 'customer_age', 'gender']], on='customer_id', how='left')

age_bins = [18, 30, 45, 60, 100]
age_labels = ['18-29 (Young Adult)', '30-44 (Mid Adult)', '45-59 (Senior Adult)', '60+ (Senior)']
claims_cust['age_group'] = pd.cut(claims_cust['customer_age'], bins=age_bins, labels=age_labels, right=False)

segment_summary = claims_cust.groupby('age_group', observed=False).agg(
    Claim_Count=('claim_id', 'count'),
    Total_Claim_Value=('claim_amount', 'sum'),
    Average_Claim_Value=('claim_amount', 'mean')
).sort_values(by='Total_Claim_Value', ascending=False)

segment_summary['Total_Claim_Value (M INR)'] = (segment_summary['Total_Claim_Value'] / 1e6).round(2)
segment_summary['Average_Claim_Value (INR)'] = segment_summary['Average_Claim_Value'].round(2)

display(segment_summary[['Claim_Count', 'Total_Claim_Value (M INR)', 'Average_Claim_Value (INR)']])

,Claim_Count,Total_Claim_Value (M INR),Average_Claim_Value (INR)
age_group,,,
45-59 (Senior Adult),2847,827.89,290792.42
30-44 (Mid Adult),2847,819.86,287971.85
18-29 (Young Adult),2249,654.66,291087.43
60+ (Senior),2057,583.04,283441.39


What is the average claim settlement time?

In [12]:
# Calculate duration across closed/settled claims
overall_avg_days = claims_df['claim_settlement_days'].dropna().mean()
print(f"Overall Average Settlement Duration: {overall_avg_days:.1f} days\n")

settlement_by_type = claims_df.dropna(subset=['claim_settlement_days']).groupby('claim_type')['claim_settlement_days'].agg(
    Average_Days='mean',
    Median_Days='median',
    Settled_Claims_Count='count'
).sort_values(by='Average_Days', ascending=False)

display(settlement_by_type.round(1))

Overall Average Settlement Duration: 23.9 days



,Average_Days,Median_Days,Settled_Claims_Count
claim_type,,,
Theft,24.1,24.0,527
Accident,24.0,24.0,1555
Third Party Damage,24.0,24.0,686
Fire,23.5,24.0,247
Natural Disaster,23.2,25.0,244


Which claim types are most common?

In [13]:
# Claim incident type frequency
claim_types = pd.DataFrame({
    'Incident Count': claims_df['claim_type'].value_counts(),
    'Share (%)': (claims_df['claim_type'].value_counts(normalize=True) * 100).round(2)
})

display(claim_types)

,Incident Count,Share (%)
claim_type,,
Accident,4838,48.38
Third Party Damage,2022,20.22
Theft,1585,15.85
Natural Disaster,796,7.96
Fire,759,7.59


How does premium compare with claim cost?

In [14]:
# Gross Written Premium vs Total Incurred Claims & Loss Ratio
total_prem = policies_df['premium_amount'].sum()
total_claims = claims_df['claim_amount'].sum()
loss_ratio = (total_claims / total_prem) * 100

financial_comparison = pd.DataFrame({
    'Metric': ['Total Gross Premium', 'Total Claim Costs', 'Portfolio Loss Ratio'],
    'Amount': [f"₹{total_prem:,.2f}", f"₹{total_claims:,.2f}", f"{loss_ratio:.2f}%"]
})

display(financial_comparison)

,Metric,Amount
0,Total Gross Premium,"₹795,568,771.71"
1,Total Claim Costs,"₹2,885,436,440.65"
2,Portfolio Loss Ratio,362.69%


Which policy types have higher claim ratios?

In [15]:
# Aggregate claim cost by policy and join to policy dataset
claims_per_policy = claims_df.groupby('policy_id')['claim_amount'].sum().reset_index()

policy_loss = policies_df.merge(claims_per_policy, on='policy_id', how='left')
policy_loss['claim_amount'] = policy_loss['claim_amount'].fillna(0)

ratio_table = policy_loss.groupby('policy_type').agg(
    Total_Premium=('premium_amount', 'sum'),
    Total_Claims=('claim_amount', 'sum')
)

ratio_table['Claim_to_Premium_Ratio (%)'] = ((ratio_table['Total_Claims'] / ratio_table['Total_Premium']) * 100).round(2)
display(ratio_table.sort_values(by='Claim_to_Premium_Ratio (%)', ascending=False))

,Total_Premium,Total_Claims,Claim_to_Premium_Ratio (%)
policy_type,,,
Third Party,7.552694e+07,8.841049e+08,1170.58
Comprehensive,5.449636e+08,1.572127e+09,288.48
Zero Depreciation,1.750783e+08,4.292049e+08,245.15


Which age groups or vehicle groups show different claim patterns?

In [16]:
# Multi-table merge to evaluate customer age vs vehicle age
m = claims_df.merge(policies_df[['policy_id', 'vehicle_id']], on='policy_id', how='left')
m = m.merge(vehicles_df[['vehicle_id', 'vehicle_age']], on='vehicle_id', how='left')
m = m.merge(customers_df[['customer_id', 'customer_age']], on='customer_id', how='left')

m['customer_age_group'] = pd.cut(m['customer_age'], bins=[18, 30, 45, 60, 100], labels=['18-29', '30-44', '45-59', '60+'])
m['vehicle_age_group'] = pd.cut(m['vehicle_age'], bins=[0, 3, 7, 12, 50], labels=['0-3 yrs (New)', '4-7 yrs (Mid)', '8-12 yrs (Old)', '12+ yrs (Very Old)'])

print("--- Claim Distribution by Customer Age Group ---")
display(m.groupby('customer_age_group', observed=False)['claim_amount'].agg(
    Claim_Count='count',
    Avg_Claim_Amount='mean',
    Total_Claim_Amount='sum'
).round(2))

print("\n--- Claim Distribution by Vehicle Age Group ---")
display(m.groupby('vehicle_age_group', observed=False)['claim_amount'].agg(
    Claim_Count='count',
    Avg_Claim_Amount='mean',
    Total_Claim_Amount='sum'
).round(2))

--- Claim Distribution by Customer Age Group ---


,Claim_Count,Avg_Claim_Amount,Total_Claim_Amount
customer_age_group,,,
18-29,2211,291176.68,6.437916e+08
30-44,2836,288230.84,8.174227e+08
45-59,2864,290323.39,8.314862e+08
60+,1867,282675.49,5.277551e+08



--- Claim Distribution by Vehicle Age Group ---


,Claim_Count,Avg_Claim_Amount,Total_Claim_Amount
vehicle_age_group,,,
0-3 yrs (New),2008,288537.09,5.793825e+08
4-7 yrs (Mid),3945,286856.55,1.131649e+09
8-12 yrs (Old),4047,290191.47,1.174405e+09
12+ yrs (Very Old),0,NaN,0.000000e+00


Which patterns require business attention?

In [17]:
# Identify operational bottlenecks and high-exposure areas
attention_items = []

# 1. Pending claim volume
pending_count = (claims_df['claim_status'] == 'Pending').sum()
pending_pct = (pending_count / len(claims_df)) * 100
attention_items.append({
    'Business Area': 'Claim Operations',
    'Observed Pattern': f"{pending_count} claims pending ({pending_pct:.1f}% of total claims)",
    'Action Required': 'Investigate operational delays and streamline approval SLAs.'
})

# 2. Frequent claimers
repeat_claimers = (claims_df['customer_id'].value_counts() > 1).sum()
attention_items.append({
    'Business Area': 'Underwriting & Fraud Risk',
    'Observed Pattern': f"{repeat_claimers} customers have more than 1 recorded claim",
    'Action Required': 'Audit repeat claims for fraud detection or adjust deductible structures.'
})

# 3. High damage severity claims
if 'damage_severity' in claims_df.columns:
    high_sev = (claims_df['damage_severity'] == 'High').sum()
    high_sev_pct = (high_sev / len(claims_df)) * 100
    attention_items.append({
        'Business Area': 'Loss Management',
        'Observed Pattern': f"{high_sev} claims classified as High severity ({high_sev_pct:.1f}%)",
        'Action Required': 'Review garage repair networks and total loss salvage procedures.'
    })

attention_report = pd.DataFrame(attention_items)
display(attention_report)

,Business Area,Observed Pattern,Action Required
0,Claim Operations,1791 claims pending (17.9% of total claims),Investigate operational delays and streamline ...
1,Underwriting & Fraud Risk,813 customers have more than 1 recorded claim,Audit repeat claims for fraud detection or adj...
2,Loss Management,1814 claims classified as High severity (18.1%),Review garage repair networks and total loss s...
